# Ocean eddy cylindrical transform

Runnable smoke notebook for the Fortran-backed xvortices core.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import xarray as xr

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from xvortices import load_cylind, project_to_cylind, storm_relative

time = np.arange(12)
lat = np.linspace(-40, -20, 161)
lon = np.linspace(140, 180, 241)
lon2d, lat2d = np.meshgrid(lon, lat)

olon = xr.DataArray(np.linspace(150, 166, time.size), dims=('time',), coords={'time': time})
olat = xr.DataArray(-30 + 2 * np.sin(np.linspace(0, 2*np.pi, time.size)), dims=('time',), coords={'time': time})

frames = []
for x0, y0 in zip(olon.values, olat.values):
    r2 = (lon2d - x0) ** 2 + (lat2d - y0) ** 2
    sla = np.exp(-r2 / 18.0)
    ugos = -(lat2d - y0) * sla / 5.0
    vgos = (lon2d - x0) * sla / 5.0
    frames.append(np.stack([sla, ugos, vgos], axis=0))
arr = np.stack(frames, axis=0)
ds = xr.Dataset(
    {name: (('time', 'lat', 'lon'), arr[:, i]) for i, name in enumerate(['sla', 'ugos', 'vgos'])},
    coords={'time': time, 'lat': lat, 'lon': lon},
)
ds

In [ ]:
fields, lons, lats, etas = load_cylind(ds, olon=olon, olat=olat, azimNum=90, radiNum=41, radMax=5)
sla, ugos, vgos = fields
uaz, vra = project_to_cylind(ugos, vgos, etas)
uc = xr.DataArray(np.gradient(olon.values), dims=('time',), coords={'time': time})
vc = xr.DataArray(np.gradient(olat.values), dims=('time',), coords={'time': time})
uaz_rel, vra_rel = storm_relative(uc, vc, uaz, vra)

assert sla.dims == ('time', 'radi', 'azim')
assert np.isfinite(sla).any()
float(uaz_rel.mean())

In [ ]:
azimuthal_structure = xr.Dataset({
    'sla_mean': sla.mean('azim'),
    'tangential_mean': uaz_rel.mean('azim'),
    'radial_mean': vra_rel.mean('azim'),
})
azimuthal_structure